In [ ]:
# ============================================================
# Swissdox@LiRI → API → TSV.xz → DataFrame nettoyé (VS Code)
# ============================================================

import os
import time
from io import BytesIO
import re
import html

import requests
import pandas as pd
import yaml
from dotenv import load_dotenv

# -------------------------------
# 0. ENV + API KEYS
# -------------------------------

# Charge les variables d'environnement depuis .env (à créer dans ton projet)
load_dotenv()

API_KEY = os.getenv("SWISSDOX_API_KEY")
API_SECRET = os.getenv("SWISSDOX_API_SECRET")

if not API_KEY or not API_SECRET:
    raise RuntimeError("Swissdox API keys missing. Set SWISSDOX_API_KEY and SWISSDOX_API_SECRET in your .env file.")

API_BASE_URL   = "https://swissdox.linguistik.uzh.ch/api"
API_URL_QUERY  = f"{API_BASE_URL}/query"
API_URL_STATUS = f"{API_BASE_URL}/status"

HEADERS = {
    "X-API-Key": API_KEY,
    "X-API-Secret": API_SECRET,
}

from datetime import datetime

QUERY_BASE_NAME = "BuerokratieVerwaltung_2025"
QUERY_NAME = f"{QUERY_BASE_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
QUERY_COMMENT   = "Requête générée depuis VS Code"
EXPIRATION_DATE = "2025-12-31"   # ou "" si tu t'en fiches

# Période
START_DATE = "2025-01-01"
END_DATE   = "2025-12-31"

# Langues
LANGUAGES = ["de", "fr"]

# Journaux / sources
SOURCES = [
    "NZZO",
    "NNTA",
    "NNHEU",
    "ZWSO",
    "TPS",
    "NZZ",
    "TA",
    "ZWAO",
    "TPSO",
    "HEU",
    "ZWAS",
    "NZZS",
    "ZWAI",
]

# Résultats max
MAX_RESULTS = 10000

# -------------------------------
# 1. FONCTIONS DE NETTOYAGE TEXTE
# -------------------------------

def clean_text(text: str) -> str:
    """Nettoie un champ texte pour usage dans pandas."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip(' "“”„\'')
    return text.strip()

def clean_xml_swissdox(text: str) -> str:
    """Supprime les balises XML Swissdox et normalise le texte."""
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)
    text = re.sub(r"</p>", "\n", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# -------------------------------
# 2. CONSTRUCTION DU YAML
# -------------------------------

query_block = {
    "sources": SOURCES,
    "dates": [
        {
            "from": START_DATE,
            "to": END_DATE,
        }
    ],
    "languages": LANGUAGES,
}

KEYWORDS = ["Bürokratie", "bureaucratie"]

query_block["content"] = {
    "OR": [
        {
            "OR": KEYWORDS
        },
        {
            "AND": [
                "öffentliche",
                "Verwaltung",
                {
                    "OR": [
                        "Bund*",
                        "bundes*",
                        "Kanton*",
                        "kantonal*"
                    ]
                }
            ]
        },
        {
            "AND": [
                "administration",
                "publique",
                {
                    "OR": [
                        "fédéral*",
                        "federal*",
                        "federale*",
                        "cantonal*",
                        "cantonale*"
                    ]
                }
            ]
        }
    ]
}

yaml_payload = {
    "query": query_block,
    "result": {
        "format": "TSV",
        "maxResults": MAX_RESULTS,
        "columns": [
            "id",
            "pubtime",
            "medium_code",
            "medium_name",
            "rubric",
            "regional",
            "doctype",
            "doctype_description",
            "language",
            "char_count",
            "dateline",
            "head",
            "subhead",
            "content_id",
            "content",
        ],
    },
    "version": "1.2",
}

yaml_query = yaml.safe_dump(
    yaml_payload,
    sort_keys=False,
    allow_unicode=True,
)

print("===== YAML envoyé à Swissdox =====")
print(yaml_query)

# -------------------------------
# 3. ENVOI DE LA REQUÊTE /query
# -------------------------------

data = {
    "query": yaml_query,
    "name": QUERY_NAME,
    "comment": QUERY_COMMENT,
    "expirationDate": EXPIRATION_DATE,
}

r = requests.post(API_URL_QUERY, headers=HEADERS, data=data)

print("===== Réponse /query =====")
print("Status code :", r.status_code)
print("Texte :", r.text)

r.raise_for_status()
resp_json = r.json()
if resp_json.get("result") != "ok":
    raise SystemExit(f"❌ Swissdox renvoie un résultat non-ok : {resp_json}")

query_id = resp_json.get("queryId") or resp_json.get("id")
if not query_id:
    raise SystemExit(f"❌ Impossible de récupérer queryId dans la réponse : {resp_json}")

print(f"✅ Requête soumise avec succès. queryId = {query_id}")

# -------------------------------
# 4. RÉCUPÉRATION DU DOWNLOAD_URL VIA /status
# -------------------------------

print("\n⏳ Récupération du downloadUrl via /status...")

download_url = None
job_info = None

for i in range(300):
    rs = requests.get(API_URL_STATUS, headers=HEADERS)
    rs.raise_for_status()

    status_list = rs.json()
    job_info = next((job for job in status_list if job.get("id") == query_id), None)

    print("----- Statut actuel -----")
    print(job_info)

    if job_info is None:
        print("ℹ️ Job pas encore visible dans /status, nouvelle tentative...")
        time.sleep(5)
        continue

    status = job_info.get("status")
    actual = job_info.get("actualResults")
    err    = job_info.get("error")
    download_url = job_info.get("downloadUrl")

    if err:
        raise SystemExit(f"❌ Erreur Swissdox pour cette requête : {err}")

    if download_url:
        print("✅ URL de téléchargement trouvée :", download_url)
        break

    if status == "finished" and (actual == 0 or actual is None) and not download_url:
        print("ℹ️ Requête terminée mais aucun résultat (actualResults = 0).")
        break

    time.sleep(5)

if not download_url:
    raise SystemExit("❌ Aucun fichier à télécharger (downloadUrl manquant).")

# -------------------------------
# 5. TÉLÉCHARGEMENT + LECTURE TSV.XZ DIRECTE
# -------------------------------

if download_url.startswith("http"):
    download_full_url = download_url
elif download_url.startswith("/"):
    download_full_url = f"{API_BASE_URL}{download_url}"
else:
    download_full_url = f"{API_BASE_URL}/download/{download_url}"

print("\n🔻 Téléchargement depuis :", download_full_url)

r_dl = requests.get(download_full_url, headers=HEADERS)
r_dl.raise_for_status()

print("Taille du fichier téléchargé : %.2f KB" % (len(r_dl.content) / 1024))

tsv_bytes = BytesIO(r_dl.content)

df = pd.read_csv(
    tsv_bytes,
    sep="\t",
    compression="xz"
)

# Conversion de pubtime en date (YYYY-MM-DD)
if "pubtime" in df.columns:
    df["pubtime"] = pd.to_datetime(df["pubtime"].astype(str), errors="coerce", utc=True).dt.date

print("Nombre de lignes chargées :", len(df))
print("Colonnes :", df.columns.tolist())

# -------------------------------
# 6. NETTOYAGE DES TEXTES
# -------------------------------

TEXT_COLS_TO_CLEAN = [
    "medium_name",
    "rubric",
    "dateline",
    "head",
    "subhead",
]

for col in TEXT_COLS_TO_CLEAN:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

if "content" in df.columns:
    df["content"] = df["content"].apply(clean_xml_swissdox)
    df["content"] = df["content"].apply(clean_text)
else:
    print("⚠️ Colonne 'content' absente du fichier téléchargé.")

# -------------------------------
# 7. APERÇU
# -------------------------------

df.head()

In [ ]:
import re
import pandas as pd

# -------------------------------
# 1. PARAMÈTRES DE RECHERCHE
# -------------------------------

KEYWORDS = [
    "Bürokratie",
    "bureaucratie",
    "Verwaltung",
    "Administration publique",
]

keyword_pattern = re.compile(
    "|".join(re.escape(k) for k in KEYWORDS),
    flags=re.IGNORECASE
)

# Colonnes d'ID / métadonnées qu'on essaie de récupérer si elles existent
META_COLS = [
    "id",
    "pubtime",
    "medium_name",
    "rubric",
    "language",
    "head",
    "subhead",
]

# -------------------------------
# 2. FONCTION POUR DÉCOUPER EN PHRASES
# -------------------------------

def split_sentences(text: str):
    """
    Découpe un texte en phrases simples basées sur . ! ?
    (Heuristique simple mais souvent suffisante)
    """
    if not isinstance(text, str):
        return []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

# -------------------------------
# 3. BOUCLE D'EXTRACTION DES PHRASES
# -------------------------------

if "content" not in df.columns:
    raise SystemExit("❌ Colonne 'content' introuvable dans df (Swissdox).")

matched_rows = []

for idx, row in df.iterrows():
    content = row.get("content", "")
    if not isinstance(content, str) or not content.strip():
        continue

    sentences = split_sentences(content)

    for sent in sentences:
        if keyword_pattern.search(sent):
            found = sorted(set(m.group(0) for m in keyword_pattern.finditer(sent)))

            out_row = {}

            # Ajouter les métadonnées si disponibles
            for col in META_COLS:
                if col in df.columns:
                    out_row[col] = row[col]

            # Ajouter la phrase et les mots-clés trouvés
            out_row["sentence"] = sent
            out_row["matched_keywords"] = ", ".join(found)
            out_row["article_row_index"] = idx  # index de l'article source

            matched_rows.append(out_row)

print(f"Nombre total de phrases identifiées avec au moins un mot-clé : {len(matched_rows)}")

if not matched_rows:
    print("⚠️ Aucune phrase trouvée avec les mots-clés fournis.")
    df_sentences = pd.DataFrame(columns=["sentence_id", "sentence", "matched_keywords"])
else:
    df_sentences = pd.DataFrame(matched_rows)

    # ID unique de phrase
    df_sentences.insert(
        0,
        "sentence_id",
        range(1, len(df_sentences) + 1)
    )

print(df_sentences.head())
print(df_sentences.shape)

In [ ]:
import os
import pandas as pd
import torch
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# -------------------------------
# 1. Vérifier que df_sentences existe
# -------------------------------

try:
    df_sentences
except NameError:
    raise RuntimeError("Le DataFrame 'df_sentences' n'existe pas. Lance d'abord la cellule d'extraction de phrases.")

SENTENCE_COL = "sentence"

if SENTENCE_COL not in df_sentences.columns:
    raise ValueError(f"Colonne '{SENTENCE_COL}' manquante dans df_sentences.")

# -------------------------------
# 2. (Optionnel) Charger un token Hugging Face depuis .env
# -------------------------------

load_dotenv()
HF_TOKEN = os.getenv("HUGGINGFACE_HUB_TOKEN")  # optionnel pour ce modèle, mais c'est propre

model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
print(f"⏳ Chargement du modèle de sentiment : {model_name}")

tokenizer_kwargs = {}
model_kwargs = {}

if HF_TOKEN:
    tokenizer_kwargs["token"] = HF_TOKEN
    model_kwargs["token"] = HF_TOKEN

tokenizer = AutoTokenizer.from_pretrained(model_name, **tokenizer_kwargs)
model = AutoModelForSequenceClassification.from_pretrained(model_name, **model_kwargs)

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# -------------------------------
# 3. Analyse par batch
# -------------------------------

texts = df_sentences[SENTENCE_COL].fillna("").astype(str).tolist()
batch_size = 64
all_labels, all_scores = [], []

print(f"🧠 Analyse des sentiments sur {len(texts)} phrases…")

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    preds = sentiment_pipeline(batch, truncation=True, max_length=256)

    for p in preds:
        all_labels.append(p["label"])
        all_scores.append(p["score"])

df_sentences["sentiment_label"] = all_labels
df_sentences["sentiment_score"] = all_scores

print("✅ Colonnes ajoutées à df_sentences : 'sentiment_label', 'sentiment_score'")
df_sentences.head()
